# Metropolis-Hastings MCMC Sampler for Multi-modal Distributions

## Introduction

In this notebook, we implement a Metropolis-Hastings MCMC sampler from scratch and test it on challenging multi-modal distributions. This demonstrates:

1. **Understanding of Markov Chain Monte Carlo** - We build the algorithm step-by-step
2. **Convergence diagnostics** - We analyze chain mixing, acceptance rates, and convergence
3. **Challenges of multi-modal distributions** - We see how MH struggles with separated modes

## Setup

In [ ]:
import sys
sys.path.append('..')
import numpy as np
import matplotlib.pyplot as plt
from src.metropolis_hastings import MetropolisHastings
from src.target_distributions import mixture_of_gaussians_log_pdf, get_distribution
from src.utils import compute_effective_sample_size, plot_contour, plot_autocorrelation
import seaborn as sns
sns.set_style('whitegrid')

%matplotlib inline

## 1. Visualizing the Target Distribution

Let's first visualize our target: a mixture of two Gaussians with unequal weights (0.4 and 0.6).

In [ ]:
# Create a grid for visualization
x = np.linspace(-6, 6, 100)
y = np.linspace(-6, 6, 100)
X, Y = np.meshgrid(x, y)
grid_points = np.column_stack([X.ravel(), Y.ravel()])

# Compute log pdf on grid
log_pdf_values = np.array([mixture_of_gaussians_log_pdf(p) for p in grid_points])
pdf_values = np.exp(log_pdf_values - np.max(log_pdf_values))
pdf_grid = pdf_values.reshape(X.shape)

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
contour = ax.contourf(X, Y, pdf_grid, levels=50, cmap='viridis')
ax.set_xlabel('Dimension 1')
ax.set_ylabel('Dimension 2')
ax.set_title('Target Distribution: Mixture of Gaussians\n(Weighted 0.4 and 0.6)')
plt.colorbar(contour, label='Density')
plt.show()

## 2. Running the MCMC Sampler

Now we initialize and run the Metropolis-Hastings sampler with adaptive proposal tuning.

In [ ]:
# Initialize the sampler with adaptive proposal
mh_sampler = MetropolisHastings(
    target_log_pdf=mixture_of_gaussians_log_pdf,
    proposal_scale=0.5,
    n_dim=2,
    adapt_scale=True,
    target_acceptance=0.234
)

# Run sampling
samples, diagnostics = mh_sampler.sample(
    n_samples=10000,
    burn_in=2000,
    thin=5
)

print(f"Acceptance rate: {diagnostics['acceptance_rate']:.3f}")
print(f"Final proposal scale: {diagnostics['proposal_scale_final']:.3f}")
print(f"Number of samples: {len(samples)}")

## 3. Visualizing the MCMC Results

### 3.1 Trace Plots and Acceptance Rate

Trace plots show how the chain moves through the parameter space over time. Good mixing means the chain explores the full space.

In [ ]:
fig = mh_sampler.plot_chain()
plt.savefig('../results/figures/mh_trace_plots.png', dpi=300, bbox_inches='tight')
plt.show()

### 3.2 Contour Plot of Sampled Points

This shows where the chain sampled relative to the true target distribution.

In [ ]:
fig = plot_contour(
    mixture_of_gaussians_log_pdf,
    samples,
    title='MCMC Samples from Metropolis-Hastings',
    save_path='../results/figures/mh_samples_contour.png'
)
plt.show()

### 3.3 Autocorrelation Analysis

Autocorrelation tells us how dependent successive samples are. Rapid decay to 0 indicates good mixing.

In [ ]:
fig = plot_autocorrelation(samples, max_lag=100)
plt.savefig('../results/figures/mh_autocorrelation.png', dpi=300, bbox_inches='tight')
plt.show()

### 3.4 Convergence Diagnostics

Effective Sample Size (ESS) tells us how many independent samples our correlated chain is equivalent to.

In [ ]:
# Effective Sample Size
ess = compute_effective_sample_size(samples)
print(f"Effective Sample Size: {ess:.0f} (out of {len(samples)})")
print(f"ESS / Total samples: {ess/len(samples):.2%}")

# Check if ESS is reasonable
if ess > 0.5 * len(samples):
    print("✅ Good mixing: ESS > 50% of total samples")
else:
    print("⚠️ Poor mixing: ESS < 50% of total samples")

## 4. Mode Switching Analysis

Let's analyze how often the chain switches between the two modes. This is crucial for multi-modal distributions.

In [ ]:
# Define mode centers
mode1_center = np.array([-3, -3])
mode2_center = np.array([3, 3])

# Assign each sample to the nearest mode
dist_to_mode1 = np.linalg.norm(samples - mode1_center, axis=1)
dist_to_mode2 = np.linalg.norm(samples - mode2_center, axis=1)
mode_assignment = np.argmin(np.column_stack([dist_to_mode1, dist_to_mode2]), axis=1)

# Plot mode assignment over time
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))

ax1.plot(mode_assignment[:500], 'o-', markersize=3)
ax1.set_ylabel('Mode Assignment')
ax1.set_xlabel('Iteration')
ax1.set_title('Mode Switching Behavior (First 500 samples)')
ax1.set_yticks([0, 1])
ax1.set_yticklabels(['Mode 1 (-3,-3)', 'Mode 2 (3,3)'])

# Count visits to each mode
mode_counts = np.bincount(mode_assignment)
ax2.bar(['Mode 1', 'Mode 2'], mode_counts)
ax2.set_ylabel('Count')
ax2.set_title(f'Mode Visits (Ratio: {mode_counts[0]/mode_counts[1]:.2f})')

plt.tight_layout()
plt.savefig('../results/figures/mh_mode_switching.png', dpi=300, bbox_inches='tight')
plt.show()

# Expected ratio based on mixture weights
expected_ratio = 0.4 / 0.6
actual_ratio = mode_counts[0] / mode_counts[1]
print(f"\nExpected mode ratio (weight 0.4 / 0.6): {expected_ratio:.3f}")
print(f"Actual mode ratio from samples: {actual_ratio:.3f}")
if abs(actual_ratio - expected_ratio) < 0.1:
    print("✅ The chain explores both modes in proportion to their weights!")

## 5. Summary Statistics

Compute summary statistics for the sampled distribution.

In [ ]:
stats = mh_sampler.get_summary_statistics()
print("Summary Statistics:")
for dim, values in stats.items():
    print(f"\n{dim}:")
    print(f"  Mean: {values['mean']:.3f}")
    print(f"  Std: {values['std']:.3f}")
    print(f"  50% (median): {values['quantiles']['50%']:.3f}")
    print(f"  95% CI: [{values['quantiles']['2.5%']:.3f}, {values['quantiles']['97.5%']:.3f}]")

## 6. Testing Different Proposal Scales

Let's see how the proposal scale affects performance.

In [ ]:
# Test different proposal scales
scales = [0.1, 0.5, 1.0, 2.0, 5.0]
acceptance_rates = []
ess_values = []

for scale in scales:
    sampler = MetropolisHastings(
        target_log_pdf=mixture_of_gaussians_log_pdf,
        proposal_scale=scale,
        n_dim=2,
        adapt_scale=False  # Turn off adaptation for fair comparison
    )
    samples, diag = sampler.sample(
        n_samples=2000,
        burn_in=500,
        thin=1,
        progress_bar=False
    )
    acceptance_rates.append(diag['acceptance_rate'])
    ess_values.append(compute_effective_sample_size(samples))

# Plot results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(scales, acceptance_rates, 'o-', markersize=8)
ax1.axhline(y=0.234, color='r', linestyle='--', label='Target (0.234)')
ax1.set_xlabel('Proposal Scale')
ax1.set_ylabel('Acceptance Rate')
ax1.set_title('Acceptance Rate vs Proposal Scale')
ax1.legend()
ax1.set_xscale('log')

ax2.plot(scales, ess_values, 'o-', markersize=8)
ax2.set_xlabel('Proposal Scale')
ax2.set_ylabel('Effective Sample Size')
ax2.set_title('ESS vs Proposal Scale')
ax2.set_xscale('log')

plt.tight_layout()
plt.savefig('../results/figures/mh_proposal_tuning.png', dpi=300, bbox_inches='tight')
plt.show()

best_idx = np.argmax(ess_values)
print(f"\nBest proposal scale: {scales[best_idx]} with ESS = {ess_values[best_idx]:.0f}")
print(f"Target acceptance rate for optimal performance: ~0.234")

## 7. Conclusion

**Key Takeaways:**

1. **Metropolis-Hastings works** but has limitations:
   - Can explore multi-modal distributions but may struggle with mode switching
   - Requires careful tuning of the proposal scale
   - Has lower efficiency than HMC for complex distributions

2. **Diagnostic tools are essential**:
   - Trace plots show convergence and mixing
   - ESS quantifies sample efficiency
   - Autocorrelation reveals dependence structure

3. **Adaptive proposals help**:
   - Automatically tune to achieve target acceptance rate
   - Reduce the need for manual tuning

**Next Steps:**
- Compare with Gibbs sampler (notebook 2)
- Benchmark against HMC (notebook 3)
- Try more challenging distributions (banana, ring)